# Tarea 2 — Inteligencia Artificial (UDP)
## Pregunta 1: Regresión lineal y regresión logística

**Integrantes:** Fernando Ortega, Ihan San Martín

**Dataset:** Hotel Booking Demand (119.390 reservas, 32 columnas). Fuente: Antonio, Almeida & Nunes (2019), *Data in Brief*; copia pública en TidyTuesday (GitHub).

### Declaración de uso de IA generativa
Se utilizó IA generativa solo para instalar el entorno (Python, Jupyter y librerías) buscar fuentes de datos públicas y verificar que el dataset cumple el tamaño mínimo, el diseño y la solucion fue hecha por nosotros. 

## 0. Librerías

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42 

## 1. Carga de datos
Cargamos el csv directo desde una URL pública, así el notebook corre en cualquier computador sin tener que descargar nada a mano.

In [2]:
URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [4]:
nulos = df.isna().sum()
nulos[nulos > 0].sort_values(ascending=False)

company     112593
agent        16340
country        488
children         4
dtype: int64

Lo primero que vimos es que hay 4 columnas con nulos. company está casi vacía y agent también tiene harto. Además las dos son códigos/IDs de la agencia o empresa, así que no son números que signifiquen algo como cantidad.children tiene solo 4 nulos y country unos cientos.

In [5]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
    "nulos_%": (df.isna().mean() * 100).round(2),
})
resumen.sort_values("valores_unicos")

,tipo,valores_unicos,nulos_%
hotel,str,2,0.00
is_canceled,int64,2,0.00
is_repeated_guest,int64,2,0.00
arrival_date_year,int64,3,0.00
deposit_type,str,3,0.00
reservation_status,str,3,0.00
customer_type,str,4,0.00
babies,int64,5,0.00
meal,str,5,0.00
distribution_channel,str,5,0.00


Con esta tabla nos hicimos una idea de qué columnas son continuas (muchos valores distintos, como adr o lead_time) y cuáles son binarias o categóricas como hotel, is_canceled, meal.

## 2. Limpieza y preprocesamiento
Antes de separar los datos hicimos una limpieza "fila por fila", o sea, cosas que no dependen de calcular promedios ni nada del conjunto completo (así no estamos mirando los datos de prueba sin querer).

In [6]:
print("filas duplicadas:", df.duplicated().sum())

personas = df["adults"] + df["children"].fillna(0) + df["babies"]
print("reservas sin personas:", (personas == 0).sum())

print(df["adr"].describe())
print("adr negativo:", (df["adr"] < 0).sum())
print("adr igual a 0:", (df["adr"] == 0).sum())
print("adr mayor a 1000:", (df["adr"] > 1000).sum())

filas duplicadas: 31994
reservas sin personas: 180
count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64
adr negativo: 1
adr igual a 0: 1959
adr mayor a 1000: 1


Lo que encontramos aquí fue que hay casi 32 mil filas repetidas exactamente iguales. Como el dataset no tiene un ID de reserva, no hay forma de saber si son reservas distintas o copias, y si las dejamos el modelo podría "ver" la misma fila en entrenamiento y en validación, lo que inflaría los resultados. Por eso decidimos sacarlas.Tambien hay 180 reservas con 0 personas, lo que no tiene sentido, así que también las sacamos y el adr tiene un valor negativo, casi 2 mil en 0 y un valor de 5400 que claramente es un error. Nos quedamos solo con adr entre 0 y 1000 (no incluimos el cero).

In [7]:
df_limpio = df.copy()
df_limpio = df_limpio.drop_duplicates()
df_limpio["children"] = df_limpio["children"].fillna(0)
personas = df_limpio["adults"] + df_limpio["children"] + df_limpio["babies"]
df_limpio = df_limpio[personas > 0]
df_limpio = df_limpio[(df_limpio["adr"] > 0) & (df_limpio["adr"] < 1000)]

print("antes:", df.shape)
print("despues:", df_limpio.shape)

antes: (119390, 32)
despues: (85585, 32)


Después sacamos algunas columnas. reservation_status y reservation_status_date por que estas dicen si la reserva terminó en "Canceled", "Check-Out" o "No-Show", que seria como la respuesta de si se canceló o no. Si las dejamos sería hacer trampa, porque esa info solo se sabe después.
Company y agent son casi puros nulos y son IDs, no aportan como número y country tiene más de 170 países distintos, lo dejamos fuera para no complicarnos con tantas categorías.

In [8]:
df_limpio = df_limpio.drop(columns=["reservation_status", "reservation_status_date", "company", "agent", "country"])

df_limpio["hotel_city"] = (df_limpio["hotel"] == "City Hotel").astype(int)                
df_limpio["deposito_no_reemb"] = (df_limpio["deposit_type"] == "Non Refund").astype(int)
meses_verano = ["June", "July", "August"]
df_limpio["temporada_alta"] = df_limpio["arrival_date_month"].isin(meses_verano).astype(int)

df_limpio.shape

(85585, 30)

Lo que hicimos aquí fue crear 3 columnas nuevas desde las columnas de texto, porque los modelos lineales necesitan números, hotel_city si es el hotel de ciudad (1) o el resort (0), deposito_no_reemb que si la persona pagó un depósito que no le devuelven y temporada_alta que si la llegada es en junio, julio o agosto.

el tema de la temporada_alta es que el mes por sí solo es un número del 1 al 12 que no sube ni baja en línea recta con el precio, entonces lo convertimos en algo de sí o no.

## 3. División 70% entrenamiento / 15% validación / 15% prueba
Primero separamos un 70% para entrenar y el 30% restante lo partimos a la mitad con el 15% validación y 15% prueba. Desde aquí en adelante todo lo que vamos a decidir lo hacemos mirando solo el conjunto de entrenamiento.

In [9]:
train, temp = train_test_split(df_limpio, test_size=0.30, random_state=SEED)
val, test = train_test_split(temp, test_size=0.50, random_state=SEED)

print("train:", train.shape, f"({len(train)/len(df_limpio):.0%})")
print("val:  ", val.shape, f"({len(val)/len(df_limpio):.0%})")
print("test: ", test.shape, f"({len(test)/len(df_limpio):.0%})")

train: (59909, 30) (70%)
val:   (12838, 30) (15%)
test:  (12838, 30) (15%)


## 4. Variables objetivo y predictoras (mirando solo entrenamiento)
### 4.1 Y1 continua:adr

In [10]:
print(train["adr"].describe())
print()
meses = ["January","February","March","April","May","June","July","August","September","October","November","December"]
print(train.groupby("arrival_date_month")["adr"].mean().reindex(meses).round(1))

count    59909.000000
mean       108.543060
std         50.255178
min          0.260000
25%         74.400000
50%         99.070000
75%        135.000000
max        510.000000
Name: adr, dtype: float64

arrival_date_month
January       72.1
February      76.5
March         82.1
April        105.1
May          113.1
June         121.6
July         138.0
August       153.0
September    114.7
October       93.3
November      74.9
December      84.7
Name: adr, dtype: float64


Elegimos Y1 = adr que es lo que se paga en promedio por noche. Es la variable continua que más sentido tiene predecir por que tiene muchos de valores distintos, va más o menos de 0 a 500 y le sirve a un hotel saber cuánto va a pagar alguien según su reserva.

En los números se ve que la mitad de las reservas está entre 74 y 135 euros (el 25% y 75%), y que en julio y agosto el precio promedio sube harto (sobre 135) comparado con invierno (alrededor de 75). Eso nos confirmó que la idea de temporada_alta iba bien.

### 4.2 Y2 binaria: is_canceled

In [11]:
print(train["is_canceled"].value_counts())
print()
print(train["is_canceled"].value_counts(normalize=True).round(3))

is_canceled
0    43202
1    16707
Name: count, dtype: int64

is_canceled
0    0.721
1    0.279
Name: proportion, dtype: float64


Teníamos 3 opciones binarias: `hotel`, `is_canceled` e `is_repeated_guest`. Elegimos **Y2 = `is_canceled`** (1 = canceló, 0 = no canceló) porque es la más interesante de predecir: un hotel quiere saber de antemano qué reservas se van a caer. `hotel` no tiene mucho sentido predecirlo (ya se sabe en qué hotel reservaste) y `is_repeated_guest` está muy desbalanceada (casi todos son 0).

En entrenamiento cerca del 28% de las reservas se cancela, o sea, las clases no están 50/50. Hay que tenerlo en cuenta, porque un modelo que diga "no cancela" siempre ya acertaría un 72%.

### 4.3 Elección de X1, ..., X10

In [12]:
# correlacion de las columnas numericas con las dos Y (solo train!)
numericas = train.select_dtypes("number")
corr = numericas.corr()[["adr", "is_canceled"]].drop(["adr", "is_canceled"])
corr["promedio_abs"] = corr.abs().mean(axis=1)
corr.sort_values("promedio_abs", ascending=False).round(3)

,adr,is_canceled,promedio_abs
temporada_alta,0.436,0.064,0.250
children,0.362,0.063,0.213
adults,0.328,0.083,0.205
total_of_special_requests,0.150,-0.122,0.136
arrival_date_year,0.181,0.088,0.135
required_car_parking_spaces,0.039,-0.187,0.113
deposito_no_reemb,-0.045,0.164,0.104
is_repeated_guest,-0.127,-0.081,0.104
hotel_city,0.122,0.075,0.098
lead_time,0.008,0.180,0.094


Lo que hicimos aquí fue ver la correlación de cada columna numérica con las dos Y, usando solo `train`. Como tenemos que usar **las mismas 10 variables para los dos modelos**, buscamos un grupo que tuviera variables buenas para predecir el precio y otras buenas para predecir la cancelación.

Las 10 que elegimos fueron:

| # | Variable | Por qué la elegimos |
|---|---|---|
| X1 | `lead_time` | días entre que reservas y llegas; es de las que más se relaciona con cancelar (mientras más anticipada, más cancela) |
| X2 | `adults` | más adultos = habitación más grande = más caro |
| X3 | `children` | igual que adults, es la que más se correlaciona con adr |
| X4 | `total_of_special_requests` | se relaciona con las dos: el que pide cosas especiales cancela menos y paga un poco más |
| X5 | `required_car_parking_spaces` | el que pide estacionamiento casi nunca cancela |
| X6 | `hotel_city` | el hotel de ciudad es más caro en promedio y cancela más |
| X7 | `temporada_alta` | en verano el precio sube harto (lo vimos en el promedio por mes) |
| X8 | `deposito_no_reemb` | sorprendentemente, los depósitos no reembolsables se cancelan más |
| X9 | `is_repeated_guest` | los clientes que repiten pagan menos y cancelan menos |
| X10 | `booking_changes` | el que hace cambios a su reserva cancela menos |

Algunas cosas que **no** usamos a propósito:
- `arrival_date_year`: tiene correlación con adr, pero solo porque los precios subieron entre 2015 y 2017. Si el modelo se usara en otro año eso no serviría.
- `arrival_date_week_number`: es parecida a `temporada_alta` pero en número (1 a 52), y la relación con el precio no es una línea recta.
- `stays_in_week_nights` / `stays_in_weekend_nights`: tienen correlación muy baja con las dos.
- `assigned_room_type`: la habitación que te asignan se sabe al llegar al hotel, así que para predecir cancelación sería medio trampa.

In [13]:
X_cols = ["lead_time", "adults", "children", "total_of_special_requests",
          "required_car_parking_spaces", "hotel_city", "temporada_alta",
          "deposito_no_reemb", "is_repeated_guest", "booking_changes"]

X_train = train[X_cols]
X_val = val[X_cols]
X_test = test[X_cols]

y1_train, y1_val, y1_test = train["adr"], val["adr"], test["adr"]
y2_train, y2_val, y2_test = train["is_canceled"], val["is_canceled"], test["is_canceled"]

print("X_train:", X_train.shape)
X_train.describe().round(2)

X_train: (59909, 10)


,lead_time,adults,children,total_of_special_requests,required_car_parking_spaces,hotel_city,temporada_alta,deposito_no_reemb,is_repeated_guest,booking_changes
count,59909.00,59909.00,59909.00,59909.00,59909.00,59909.00,59909.00,59909.00,59909.00,59909.00
mean,80.83,1.88,0.14,0.70,0.08,0.61,0.33,0.01,0.03,0.26
std,86.06,0.50,0.46,0.83,0.28,0.49,0.47,0.11,0.18,0.70
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,12.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,51.00,2.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00
75%,126.00,2.00,0.00,1.00,0.00,1.00,1.00,0.00,0.00,0.00
max,709.00,4.00,3.00,5.00,8.00,1.00,1.00,1.00,1.00,18.00


### 4.4 Escalado
Las variables tienen escalas muy distintas (`lead_time` llega a más de 600 y las binarias son 0 o 1). Para la regresión logística esto importa porque usa regularización, y además así los coeficientes se pueden comparar entre ellos. El `StandardScaler` lo **ajustamos solo con train** y después lo aplicamos a val y test con esos mismos valores.

In [14]:
scaler = StandardScaler()
scaler.fit(X_train)   # solo con train

X_train_esc = scaler.transform(X_train)
X_val_esc = scaler.transform(X_val)
X_test_esc = scaler.transform(X_test)

## 5. Selección de variables (solo con entrenamiento): top 5 y top 3
_Justificación del método:_

## 6. Tarea A: regresión lineal para Y1 (10, 5 y 3 variables)

### 6.1 Comparación entrenamiento vs validación

### 6.2 Evaluación en prueba de las 2 mejores versiones

## 7. Tarea B: regresión logística para Y2 (10, 5 y 3 variables)

### 7.1 Comparación entrenamiento vs validación

### 7.2 Evaluación en prueba de las 2 mejores versiones

## 8. Análisis: cantidad de variables, generalización, overfitting/underfitting

_(redactar)_